# 3단계: 문서 리트리버 (Colab, GPU)

claim에 필요한 위키 문서를 찾는다. 두 단계로 나눈다.
1. **TF-IDF 1차 검색**: HoVer 저자들이 미리 계산한 claim별 top-100 문서를 그대로 쓴다 (0단계에서 받아 둠)
2. **재정렬(rerank)**: 상위 `K_CAND`(=20, 원본과 같음)개를 roberta-base가 (claim, 문서) 쌍마다 "정답 문서일 확률"로 다시 매긴다

평가는 **recall@k**: 정답 문서 2개가 top-k 안에 **모두** 들어온 claim의 비율.
TF-IDF top-20 안에 정답이 다 들어오는 비율(test 58%)이 재정렬의 천장이다. 재정렬은 순서만 바꾸고 새 문서를 가져오지 못한다.

| 셀 | 역할 |
|---|---|
| 1 | 데이터, TF-IDF 결과, 위키 DB 불러오기 |
| 2 | TF-IDF만 썼을 때 recall@k (재정렬 전 기준) |
| 3 | (claim, 문서) 쌍 만들기. 학습 때는 원본처럼 정답 문서를 항상 넣는다 |
| 4 | 재정렬 모델 학습 (쌍 약 16만 개, 1 epoch) |
| 5 | 재정렬 후 recall@k, TF-IDF와 비교 |
| 6 | 재정렬 결과 저장 (4단계 문장 선택에서 씀) |

In [10]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U transformers datasets accelerate

import json, os, sqlite3, unicodedata
from functools import lru_cache
import numpy as np, pandas as pd

ROOT = '/content/drive/MyDrive/fact-verification'
RAW, PROC = f'{ROOT}/data/raw', f'{ROOT}/data/processed'
WIKI = '/content/wiki_wo_links.db'
if not os.path.exists(WIKI):
    !cp "{RAW}/wiki_wo_links.db" {WIKI}
con = sqlite3.connect(WIKI)

dfs = {n: pd.read_json(f'{PROC}/hover2_{n}.jsonl', lines=True) for n in ['train', 'val', 'test']}
nfc = lambda s: unicodedata.normalize('NFC', s)  # 정답 제목은 NFC, 위키 DB/TF-IDF 제목은 NFD라서 비교 전에 통일
tfidf = {}
for split in ['train', 'dev']:  # train/val은 train 파일, test(공식 dev)는 dev 파일에 있다
    for e in json.load(open(f'{RAW}/{split}_tfidf_doc_retrieval_results.json')):
        tfidf[e['id']] = e['doc_retrieval_results'][0][0]  # 점수 순 문서 제목 100개
for d in dfs.values():
    d['gold_docs'] = d.supporting_facts.map(lambda sf: list(dict.fromkeys(nfc(t) for t, _ in sf)))
    d['cands'] = d.uid.map(lambda u: [nfc(t) for t in tfidf[u]])
{n: len(d) for n, d in dfs.items()}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


{'train': 8166, 'val': 886, 'test': 1126}

In [11]:
K_CAND = 20
KS = [2, 5, 8, 10, 20]  # 5, 8, 10은 HoVer 원본 논문과 같은 k

def recall_at_k(ranked, golds, k):
    return float(np.mean([set(g) <= set(r[:k]) for r, g in zip(ranked, golds)]))

for n in ['val', 'test']:
    d = dfs[n]
    print(n, 'TF-IDF', {k: round(recall_at_k(d.cands, d.gold_docs, k), 3) for k in KS + [100]})

val TF-IDF {2: 0.282, 5: 0.466, 8: 0.552, 10: 0.585, 20: 0.677, 100: 0.801}
test TF-IDF {2: 0.244, 5: 0.397, 8: 0.469, 10: 0.51, 20: 0.602, 100: 0.775}


In [12]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL = 'roberta-base'
MAX_LEN = 256
tok = AutoTokenizer.from_pretrained(MODEL)

@lru_cache(maxsize=None)
def doc_text(title):
    row = con.execute('SELECT text FROM documents WHERE id = ?', (unicodedata.normalize('NFD', title),)).fetchone()
    return row[0] if row else ''

def make_pairs(d, train):
    rows = []
    for r in d.itertuples():
        titles = list(r.cands[:K_CAND])
        if train:
            # TF-IDF가 정답을 놓친 claim도 양성 예시가 있도록 정답 문서를 앞에 넣는다 (원본 HoVer 방식)
            titles = (r.gold_docs + [t for t in titles if t not in r.gold_docs])[:K_CAND]
        rows += [{'uid': r.uid, 'claim': r.claim, 'title': t, 'labels': int(t in r.gold_docs)} for t in titles]
    return pd.DataFrame(rows)

pairs = {n: make_pairs(d, n == 'train') for n, d in dfs.items()}

def encode(b):
    # 제목에 구분 정보가 있어서(예: "Life Goes On (Fergie song)") 본문 앞에 붙인다
    return tok(b['claim'], [f'{t}. {doc_text(t)}' for t in b['title']], truncation='only_second', max_length=MAX_LEN)

ds = {n: Dataset.from_pandas(p[['claim', 'title', 'labels']]).map(encode, batched=True, remove_columns=['claim', 'title'])
      for n, p in pairs.items()}
{n: (len(p), round(p.labels.mean(), 3)) for n, p in pairs.items()}  # (쌍 개수, 양성 비율)

Map:   0%|          | 0/163320 [00:00<?, ? examples/s]

Map:   0%|          | 0/17713 [00:00<?, ? examples/s]

Map:   0%|          | 0/22520 [00:00<?, ? examples/s]

{'train': (163320, np.float64(0.1)),
 'val': (17713, np.float64(0.082)),
 'test': (22520, np.float64(0.077))}

In [13]:
import torch
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
args = TrainingArguments(
    output_dir='/content/out/doc_rerank',
    num_train_epochs=1, learning_rate=2e-5, per_device_train_batch_size=32, per_device_eval_batch_size=128,
    warmup_steps=0.1, weight_decay=0.01, fp16=True, seed=42,
    save_strategy='no', logging_steps=200, report_to='none',
)
# 순위만 쓰므로 클래스 가중치는 쓰지 않는다 (가중치는 확률의 절대값을 옮길 뿐 순서는 거의 그대로)
trainer = Trainer(model=model, args=args, train_dataset=ds['train'], data_collator=DataCollatorWithPadding(tok))
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
200,0.449082
400,0.251588
600,0.172525
800,0.141448
1000,0.129238
1200,0.133961
1400,0.140472
1600,0.110411
1800,0.117108
2000,0.112650


TrainOutput(global_step=5104, training_loss=0.12604939732058296, metrics={'train_runtime': 1842.6662, 'train_samples_per_second': 88.632, 'train_steps_per_second': 2.77, 'total_flos': 2.137331680449984e+16, 'train_loss': 0.12604939732058296, 'epoch': 1.0})

In [14]:
# 웹 데모(app.py)에서 쓰도록 학습한 모델을 Drive에 저장한다
SAVE = '/content/drive/MyDrive/fact-verification/models/doc_rerank'
trainer.save_model(SAVE); tok.save_pretrained(SAVE)
!ls -lh {SAVE}

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

total 479M
-rw------- 1 root root  740 Oct 25 06:08 config.json
-rw------- 1 root root 476M Oct 25 06:08 model.safetensors
-rw------- 1 root root  388 Oct 25 06:08 tokenizer_config.json
-rw------- 1 root root 3.4M Oct 25 06:08 tokenizer.json
-rw------- 1 root root 5.1K Oct 25 06:08 training_args.bin


In [15]:
def rerank(n):
    p = pairs[n].copy()
    logits = trainer.predict(ds[n]).predictions
    p['score'] = torch.softmax(torch.tensor(logits), -1)[:, 1].numpy()
    p = p.sort_values('score', ascending=False)
    g = p.groupby('uid', sort=False)  # 그룹 안의 순서는 정렬된 순서를 유지한다
    return dfs[n].uid.map(g.title.apply(list)), dfs[n].uid.map(g.score.apply(list))

for n in ['train', 'val', 'test']:
    dfs[n]['reranked'], dfs[n]['rerank_scores'] = rerank(n)

for n in ['val', 'test']:
    d = dfs[n]
    print(n)
    print('  TF-IDF  ', {k: round(recall_at_k(d.cands, d.gold_docs, k), 3) for k in KS})
    print('  rerank  ', {k: round(recall_at_k(d.reranked, d.gold_docs, k), 3) for k in KS})

val
  TF-IDF   {2: 0.282, 5: 0.466, 8: 0.552, 10: 0.585, 20: 0.677}
  rerank   {2: 0.6, 5: 0.665, 8: 0.674, 10: 0.677, 20: 0.677}
test
  TF-IDF   {2: 0.244, 5: 0.397, 8: 0.469, 10: 0.51, 20: 0.602}
  rerank   {2: 0.511, 5: 0.577, 8: 0.595, 10: 0.599, 20: 0.602}


In [16]:
for n, d in dfs.items():
    d[['uid', 'reranked', 'rerank_scores']].to_json(f'{PROC}/hover2_{n}_docret.jsonl', orient='records', lines=True, force_ascii=False)
!ls -lh "{PROC}" 

total 61M
-rw------- 1 root root 883K Oct 25 06:20 hover2_test_docret.jsonl
-rw------- 1 root root 547K Oct 21 14:02 hover2_test_docret_mh.jsonl
-rw------- 1 root root 2.4M Oct 21 13:28 hover2_test_e2e_retrieved.jsonl
-rw------- 1 root root 2.3M Oct 23 02:26 hover2_test_e2e_retrieved_mh.jsonl
-rw------- 1 root root 768K Oct 21 06:22 hover2_test.jsonl
-rw------- 1 root root 2.1M Oct 21 12:55 hover2_test_sentret.jsonl
-rw------- 1 root root 2.0M Oct 22 13:14 hover2_test_sentret_mh.jsonl
-rw------- 1 root root 6.3M Oct 25 06:20 hover2_train_docret.jsonl
-rw------- 1 root root 4.2M Oct 21 14:02 hover2_train_docret_mh.jsonl
-rw------- 1 root root 5.5M Oct 21 06:22 hover2_train.jsonl
-rw------- 1 root root  15M Oct 21 12:55 hover2_train_sentret.jsonl
-rw------- 1 root root  15M Oct 22 13:14 hover2_train_sentret_mh.jsonl
-rw------- 1 root root 696K Oct 25 06:20 hover2_val_docret.jsonl
-rw------- 1 root root 430K Oct 21 14:02 hover2_val_docret_mh.jsonl
-rw------- 1 root root 612K Oct 21 06:22 